In [1]:
#pip install trl

# SFT

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from IPython.display import display, HTML, Markdown
import torch
from trl import SFTConfig, SFTTrainer
import math

model_id = "LiquidAI/LFM2-700M" # <- or LFM2-700M or LFM2-350M
normalized_model_id = model_id.replace("/", "_").replace("-", "_").replace(".", "_")

/home/igonzalez/env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
torch.cuda.is_available()

True

In [4]:
#pip uninstall -y torch torchvision torchaudio

In [5]:
#pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126

In [6]:
import sys
import torch

print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

Python: /home/igonzalez/env/bin/python3
PyTorch: 2.13.0+cu130
CUDA build: 13.0
CUDA available: False
GPU count: 1


In [7]:
!nvidia-smi

Mon Jul 20 21:13:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3090        Off |   00000000:03:00.0 Off |                  N/A |
|  0%   40C    P8             31W /  370W |    4184MiB /  24576MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [6]:
print("📚 Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id)

📚 Loading tokenizer...


In [7]:
print("🧠 Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype="bfloat16",
    # attn_implementation="flash_attention_2" # <- uncomment on compatible GPU
    attn_implementation="sdpa",
)

print("✅ Local model loaded successfully!")
print(f"🔢 Parameters: {model.num_parameters():,}")
print(f"📖 Vocab size: {len(tokenizer)}")
print(f"💾 Model size: ~{model.num_parameters() * 2 / 1e9:.1f} GB (bfloat16)")

🧠 Loading model...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 148/148 [00:00<00:00, 445.12it/s]


✅ Local model loaded successfully!
🔢 Parameters: 742,489,344
📖 Vocab size: 64400
💾 Model size: ~1.5 GB (bfloat16)


In [8]:
from datasets import load_dataset, Dataset

print("📥 Loading SFT dataset...")
dataset = load_dataset("thinkPy/paraguay-cultural-alignment", "sft")

📥 Loading SFT dataset...


Generating validation split: 100%|██████████| 300/300 [00:00<00:00, 49313.81 examples/s]


In [9]:
dataset

DatasetDict({
    train: Dataset({
        features: ['prompt', 'response', 'topic', 'cultural', 'score'],
        num_rows: 2700
    })
    validation: Dataset({
        features: ['prompt', 'response', 'topic', 'cultural', 'score'],
        num_rows: 300
    })
})

In [10]:
def format(example):
    return {
        "messages": [
            {"role": "user", "content": example['prompt']},
            {"role": "assistant", "content": example['response']}
        ]
    }

dataset_sft = dataset.map(format, remove_columns=dataset["train"].column_names)

Map: 100%|██████████| 300/300 [00:00<00:00, 7737.24 examples/s]


In [11]:
dataset_sft

DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 2700
    })
    validation: Dataset({
        features: ['messages'],
        num_rows: 300
    })
})

In [12]:
train_dataset_sft = dataset_sft["train"]
val_dataset_sft = dataset_sft["validation"]

print("✅ SFT Dataset loaded:")
print(f"   📚 Train samples: {len(train_dataset_sft)}")
print(f"   🧪 Eval samples: {len(val_dataset_sft)}")
print(f"\n📝 Single Sample: {train_dataset_sft[0]['messages']}")

✅ SFT Dataset loaded:
   📚 Train samples: 2700
   🧪 Eval samples: 300

📝 Single Sample: [{'role': 'user', 'content': 'Bailar bajo la lluvia, la melodía comienza,\nUna sinfonía de precipitación, música en nuestros corazones,\nEl golpeteo de las gotas de lluvia, llena el aire,\nEl sonido de la naturaleza, una orquesta tan rara.\n\nLas gotas de lluvia golpean el pavimento y cantan,\nUna dulce percusión, el sonido que trae,\nEl ritmo de la lluvia al caer sobre los techos,\nEs como un ritmo de tambor, un sonido que ladra.\n\nGolpea contra las hojas de los árboles,\nUna melodía como el susurro de las llaves,\nComo si la lluvia estuviera tocando una melodía,\nUna armonía que nos deja en un estado de éxtasis.\n\nEl sonido de la lluvia en diferentes superficies,\nEs como una danza de música, emerge,\nUna sinfonía acústica, la lluvia y nosotros,\nUn medley melodioso, un coro divino.\n\nY mientras bailamos, la música a nuestro alrededor se eleva,\nUna celebración de la vida, un momento que sobres

In [13]:
non_reasoning_dataset = train_dataset_sft.map(lambda x: {"text": tokenizer.apply_chat_template(
    x["messages"],
    tokenize = False,
)})

Map: 100%|██████████| 2700/2700 [00:00<00:00, 4395.24 examples/s]


In [14]:
non_reasoning_dataset[0]

{'messages': [{'role': 'user',
   'content': 'Bailar bajo la lluvia, la melodía comienza,\nUna sinfonía de precipitación, música en nuestros corazones,\nEl golpeteo de las gotas de lluvia, llena el aire,\nEl sonido de la naturaleza, una orquesta tan rara.\n\nLas gotas de lluvia golpean el pavimento y cantan,\nUna dulce percusión, el sonido que trae,\nEl ritmo de la lluvia al caer sobre los techos,\nEs como un ritmo de tambor, un sonido que ladra.\n\nGolpea contra las hojas de los árboles,\nUna melodía como el susurro de las llaves,\nComo si la lluvia estuviera tocando una melodía,\nUna armonía que nos deja en un estado de éxtasis.\n\nEl sonido de la lluvia en diferentes superficies,\nEs como una danza de música, emerge,\nUna sinfonía acústica, la lluvia y nosotros,\nUn medley melodioso, un coro divino.\n\nY mientras bailamos, la música a nuestro alrededor se eleva,\nUna celebración de la vida, un momento que sobresale,\nEl sonido de las gotas de lluvia, una melodía dulce,\nUn baile baj

In [15]:
num_train_epochs = 3
per_device_train_batch_size = 2
per_device_eval_batch_size = 1
gradient_accumulation_steps = 4
num_devices=1

num_train_examples = len(train_dataset_sft)
num_eval_examples = len(val_dataset_sft)

# Batch size efectivo de entrenamiento
effective_train_batch_size = (
    per_device_train_batch_size
    * gradient_accumulation_steps
    * num_devices
)

# Batch size efectivo de evaluación
effective_eval_batch_size = (
    per_device_eval_batch_size
    * num_devices
)

# Steps de entrenamiento
steps_per_epoch = math.ceil(
    num_train_examples / effective_train_batch_size
)

total_training_steps = steps_per_epoch * num_train_epochs

# Steps de evaluación: cuántos batches tiene validation
eval_steps_per_run = math.ceil(
    num_eval_examples / effective_eval_batch_size
)

print("Train examples:", num_train_examples)
print("Validation examples:", num_eval_examples)
print()
print("Train effective batch size:", effective_train_batch_size)
print("Eval effective batch size:", effective_eval_batch_size)
print()
print("Steps por epoch:", steps_per_epoch)
print("Total training steps:", total_training_steps)
print("Eval batches por evaluación:", eval_steps_per_run)

Train examples: 2700
Validation examples: 300

Train effective batch size: 8
Eval effective batch size: 1

Steps por epoch: 338
Total training steps: 1014
Eval batches por evaluación: 300


In [16]:
import os

tb_log_dir = f"./train/sft/{normalized_model_id}/logs"
os.environ["TENSORBOARD_LOGGING_DIR"] = tb_log_dir

sft_config = SFTConfig(
    output_dir=f"./train/{normalized_model_id}",

    num_train_epochs=num_train_epochs,

    per_device_train_batch_size=per_device_train_batch_size,
    per_device_eval_batch_size=per_device_eval_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,

    learning_rate=5e-5,
    lr_scheduler_type="cosine",

    warmup_steps=math.ceil(total_training_steps * 0.01),

    logging_strategy="steps",
    logging_steps=math.ceil(total_training_steps * 0.05),

    save_strategy="steps",
    save_steps=math.ceil(total_training_steps * 0.05),#save_steps,
    save_total_limit=3,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    eval_strategy="steps",
    eval_steps=math.ceil(total_training_steps * 0.05),

    report_to=["tensorboard"],

    dataset_text_field="messages",
    bf16=True,
)

In [17]:
pip install tensorboard --break-system-packages


[notice] A new release of pip is available: 23.1.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [18]:
print("🏗️  Creating SFT trainer...")
sft_trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset_sft,
    eval_dataset=val_dataset_sft,
    processing_class=tokenizer,
)

print("\n🚀 Starting SFT training...")
sft_trainer.train()

print("🎉 SFT training completed!")

🏗️  Creating SFT trainer...


Truncating eval dataset: 100%|██████████| 300/300 [00:00<00:00, 1971.60 examples/s]
2026-07-20 21:28:40.682219: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-07-20 21:28:41.923838: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT



🚀 Starting SFT training...


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
51,1.954051,1.603899,1.636840,188783.000000,0.637134
102,1.453643,1.375738,1.395084,375384.000000,0.682911
153,1.235846,1.242701,1.273523,566056.000000,0.712601
204,1.151398,1.128800,1.196273,755603.000000,0.736597
255,0.998941,1.016094,1.032764,940878.000000,0.762344
306,0.917467,0.937687,1.027558,1125089.000000,0.780827
357,0.731430,0.904913,0.846035,1313263.000000,0.792799
408,0.497265,0.862574,0.782715,1497742.000000,0.803345
459,0.507775,0.831577,0.774685,1686349.000000,0.812336
510,0.464783,0.818443,0.722106,1875754.000000,0.818262


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.81s/it]
[transformers] There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


🎉 SFT training completed!


In [19]:
pip install python-dotenv --break-system-packages


[notice] A new release of pip is available: 23.1.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [20]:
pip install ipywidgets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 2.1 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 21.2 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 914.9/914.9 kB 22.6 MB/s eta 0:00:0000:01

[notice] A new release of pip is available: 23.1.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [21]:
import os
from dotenv import load_dotenv
from huggingface_hub import login, HfApi
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig

'''load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")

login(token=HF_TOKEN)

#https://huggingface.co/LiquidAI/LFM2.5-350M
# Repo destino
hub_model_id = "thinkPy/sft_LFM2_5_350M"'''

# Usa tu mejor checkpoint DPO
best_checkpoint_path = "./train/LiquidAI_LFM2_700M/checkpoint-1014"
# Si tu checkpoint real tiene otro número, cámbialo aquí.

# Cargar modelo y tokenizer desde el checkpoint
model = AutoModelForCausalLM.from_pretrained(
    best_checkpoint_path,
    device_map="auto",
    dtype="bfloat16",
)

'''best_checkpoint_path = os.path.abspath("./train/sft/LiquidAI_LFM2_5_350M/checkpoint-1014/")

model = AutoModelForCausalLM.from_pretrained(
    best_checkpoint_path,
    device_map="auto",
    dtype="bfloat16",
)'''

'''best_checkpoint_path = os.path.abspath("./train/sft/LiquidAI_LFM2_5_350M/checkpoint-1014")

config = AutoConfig.from_pretrained(best_checkpoint_path)

model = AutoModelForCausalLM.from_pretrained(
    best_checkpoint_path,
    config=config,
    device_map="auto",
    torch_dtype="bfloat16",   # <-- nota: es torch_dtype, no dtype
    local_files_only=True,    # fuerza a buscar solo localmente
)'''

#tokenizer = AutoTokenizer.from_pretrained(best_checkpoint_path, local_files_only=True)

tokenizer = AutoTokenizer.from_pretrained(best_checkpoint_path)


# Recomendado para inferencia
model.config.use_cache = True

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 476.67it/s]


In [ ]:
import os

'''model.push_to_hub(
    hub_model_id,
    token=HF_TOKEN,
    private=False,
)

tokenizer.push_to_hub(
    hub_model_id,
    token=HF_TOKEN,
    private=False,
)

print(f"✅ Modelo subido a: https://huggingface.co/{hub_model_id}")'''

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.53it/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (0 / 1):   0%|          |  611kB /  709MB, 57.3kB/s  
Processing Files (0 / 1):   0%|          | 1.85MB /  709MB,  170kB/s  
Processing Files (0 / 1):   0%|          | 3.07MB /  709MB,  280kB/s  
Processing Files (0 / 1):   1%|          | 3.69MB /  709MB,  329kB/s  
Processing Files (0 / 1):   1%|          | 4.94MB /  709MB,  440kB/s  
Processing Files (0 / 1):   1%|          | 6.16MB /  709MB,  547kB/s  
Processing Files (0 / 1):   1%|          | 7.39MB /  709MB,  634kB/s  
Processing Files (0 / 1):   1%|▏         | 9.25MB /  709MB,  778kB/s  
Processing Files (0 / 1):   2%|▏         | 11.1MB /  709MB,  914kB/s  
Processing Files (0 / 1):   2%|▏         | 11.7MB /  709MB,  935kB/s  
Processing Files (0 / 1):   2%|▏         | 13.0MB /  709MB, 1.04MB/s  
Processing Files (0 / 1):   2%|▏         | 14.8MB /  709MB, 1.16MB/s  
Processing Fi

✅ Modelo subido a: https://huggingface.co/thinkPy/sft_LFM2_5_350M


# DPO

In [22]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from IPython.display import display, HTML, Markdown
import torch
from trl import SFTConfig, SFTTrainer
import math

model_id = "LiquidAI/LFM2-700M"
normalized_model_id = model_id.replace("/", "_").replace("-", "_").replace(".", "_")
load_model_id = f"./train/{normalized_model_id}/checkpoint-1014"

In [23]:
print("📚 Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(load_model_id)

📚 Loading tokenizer...


In [24]:
print("🧠 Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    load_model_id,
    device_map="auto",
    dtype="bfloat16",
    # attn_implementation="flash_attention_2" # <- uncomment on compatible GPU
    attn_implementation="sdpa",
)

print("✅ Local model loaded successfully!")
print(f"🔢 Parameters: {model.num_parameters():,}")
print(f"📖 Vocab size: {len(tokenizer)}")
print(f"💾 Model size: ~{model.num_parameters() * 2 / 1e9:.1f} GB (bfloat16)")

🧠 Loading model...


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 447.14it/s]


✅ Local model loaded successfully!
🔢 Parameters: 742,489,344
📖 Vocab size: 64400
💾 Model size: ~1.5 GB (bfloat16)


In [25]:
from datasets import load_dataset, Dataset

print("📥 Loading SFT dataset...")
dataset = load_dataset("thinkPy/paraguay-cultural-alignment", "dpo")

📥 Loading SFT dataset...


Generating test split: 100%|██████████| 300/300 [00:00<00:00, 48663.46 examples/s]


In [26]:
dataset

DatasetDict({
    train: Dataset({
        features: ['prompt', 'chosen', 'rejected', 'topic', 'cultural_chosen', 'cultural_rejected', 'score_chosen', 'score_rejected'],
        num_rows: 1500
    })
    validation: Dataset({
        features: ['prompt', 'chosen', 'rejected', 'topic', 'cultural_chosen', 'cultural_rejected', 'score_chosen', 'score_rejected'],
        num_rows: 200
    })
    test: Dataset({
        features: ['prompt', 'chosen', 'rejected', 'topic', 'cultural_chosen', 'cultural_rejected', 'score_chosen', 'score_rejected'],
        num_rows: 300
    })
})

In [27]:
def format_dpo_plain(example):
    prompt = example["prompt"].strip()
    chosen = example["chosen"].strip()
    rejected = example["rejected"].strip()

    return {
        "prompt": prompt + " \n",
        "chosen": chosen,
        "rejected": rejected,
    }

dataset_dpo = dataset.map(format_dpo_plain)
dataset_dpo = dataset_dpo.select_columns(["prompt", "chosen", "rejected"])

Map: 100%|██████████| 300/300 [00:00<00:00, 5330.45 examples/s]


In [28]:
dataset_dpo

DatasetDict({
    train: Dataset({
        features: ['prompt', 'chosen', 'rejected'],
        num_rows: 1500
    })
    validation: Dataset({
        features: ['prompt', 'chosen', 'rejected'],
        num_rows: 200
    })
    test: Dataset({
        features: ['prompt', 'chosen', 'rejected'],
        num_rows: 300
    })
})

In [29]:
dataset_dpo["train"][0]

{'prompt': '¡Muchas gracias por sus amables palabras! Me alegra que disfrutara del poema. En cuanto a los hombres lobo, eran en efecto una presencia amenazante en el bosque encantado. Su pelaje era tan negro como el cielo nocturno, y sus ojos brillaban como bolas de fuego, advirtiendo al aventurero de su hambre y sed de sangre implacables.\n\nSu comportamiento era el de depredadores alfa, astutos y brutales en sus ataques. Acechaban a sus presas con una gracia feral, arrastrándose silenciosamente por la maleza hasta que estaban lo suficientemente cerca como para saltar. Una vez que atacaban, eran implacables, golpeando con toda la fuerza de sus garras y dientes afilados como navajas.\n\nEl aventurero fue sorprendido por su ferocidad, pero no fue fácilmente derrotado. Con su espada en la mano, luchó con todas sus fuerzas, esquivando y zigzagueando entre los árboles mientras los hombres lobo se lanzaban hacia él desde diferentes direcciones.\n\nSu batalla fue agotadora, ya que los hombre

In [30]:
train_dataset_dpo = dataset_dpo["train"]
val_dataset_dpo = dataset_dpo["validation"]
test_dataset_dpo = dataset_dpo["test"]

print("✅ DPO Dataset loaded:")
print(f"   📚 Train samples: {len(train_dataset_dpo)}")
print(f"   🧪 Eval samples: {len(val_dataset_dpo)}")
print(f"\n📝 Single Sample: {train_dataset_dpo[0]['prompt']}")

✅ DPO Dataset loaded:
   📚 Train samples: 1500
   🧪 Eval samples: 200

📝 Single Sample: ¡Muchas gracias por sus amables palabras! Me alegra que disfrutara del poema. En cuanto a los hombres lobo, eran en efecto una presencia amenazante en el bosque encantado. Su pelaje era tan negro como el cielo nocturno, y sus ojos brillaban como bolas de fuego, advirtiendo al aventurero de su hambre y sed de sangre implacables.

Su comportamiento era el de depredadores alfa, astutos y brutales en sus ataques. Acechaban a sus presas con una gracia feral, arrastrándose silenciosamente por la maleza hasta que estaban lo suficientemente cerca como para saltar. Una vez que atacaban, eran implacables, golpeando con toda la fuerza de sus garras y dientes afilados como navajas.

El aventurero fue sorprendido por su ferocidad, pero no fue fácilmente derrotado. Con su espada en la mano, luchó con todas sus fuerzas, esquivando y zigzagueando entre los árboles mientras los hombres lobo se lanzaban hacia él desd

In [31]:
num_train_epochs = 1
per_device_train_batch_size = 1
per_device_eval_batch_size = 1
gradient_accumulation_steps = 4
num_devices=1

num_train_examples = len(train_dataset_dpo)
num_eval_examples = len(val_dataset_dpo)

# Batch size efectivo de entrenamiento
effective_train_batch_size = (
    per_device_train_batch_size
    * gradient_accumulation_steps
    * num_devices
)

# Batch size efectivo de evaluación
effective_eval_batch_size = (
    per_device_eval_batch_size
    * num_devices
)

# Steps de entrenamiento
steps_per_epoch = math.ceil(
    num_train_examples / effective_train_batch_size
)

total_training_steps = steps_per_epoch * num_train_epochs

# Steps de evaluación: cuántos batches tiene validation
eval_steps_per_run = math.ceil(
    num_eval_examples / effective_eval_batch_size
)

warmup_steps=math.ceil(total_training_steps * 0.03)
logging_steps=math.ceil(total_training_steps * 0.01)
eval_steps=math.ceil(total_training_steps * 0.05)
save_steps=math.ceil(total_training_steps * 0.05)

print("Train examples:", num_train_examples)
print("Validation examples:", num_eval_examples)
print()
print("Train effective batch size:", effective_train_batch_size)
print("Eval effective batch size:", effective_eval_batch_size)
print()
print("Steps por epoch:", steps_per_epoch)
print("Total training steps:", total_training_steps)
print("Eval batches por evaluación:", eval_steps_per_run)
print()
print("Warmup Steps:", warmup_steps)
print("Logging Steps:", logging_steps)
print("Eval Steps:", eval_steps)
print("Save Steps:", save_steps)

Train examples: 1500
Validation examples: 200

Train effective batch size: 4
Eval effective batch size: 1

Steps por epoch: 375
Total training steps: 375
Eval batches por evaluación: 200

Warmup Steps: 12
Logging Steps: 4
Eval Steps: 19
Save Steps: 19


In [32]:
import os
from trl import DPOConfig, DPOTrainer

# model.config.use_cache = False

output_dir = f"./train/dpo/{normalized_model_id}"
os.environ["TENSORBOARD_LOGGING_DIR"] = f"{output_dir}/logs"

dpo_config = DPOConfig(
    output_dir=output_dir,

    num_train_epochs=num_train_epochs,

    per_device_train_batch_size=per_device_train_batch_size,
    per_device_eval_batch_size=per_device_eval_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,

    # DPO suele usar LR mucho más bajo que SFT
    learning_rate=1e-6,
    lr_scheduler_type="cosine",
    warmup_steps=warmup_steps,

    # DPO estándar
    beta=0.1,
    loss_type="sigmoid",

    # Logging / eval / checkpoints
    logging_strategy="steps",
    logging_steps=logging_steps,

    eval_strategy="steps",
    eval_steps=eval_steps,

    save_strategy="steps",
    save_steps=save_steps,
    save_total_limit=3,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # gradient_checkpointing=True,

    report_to=["tensorboard"],

    bf16=True,
)

In [33]:
print("🏗️ Creating DPO trainer...")

dpo_trainer = DPOTrainer(
    model=model,
    args=dpo_config,
    train_dataset=train_dataset_dpo,
    eval_dataset=val_dataset_dpo,
    processing_class=tokenizer,
)


🏗️ Creating DPO trainer...


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 177.19it/s]


In [34]:
print("\n🚀 Starting DPO training...")
train_result = dpo_trainer.train()

# model.config.use_cache = True

print("🎉 DPO training completed!")


🚀 Starting DPO training...


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
19,0.680322,0.670667,0.955016,68055.000000,-1.766678,-1.954936,0.720600,0.000062,-0.048047,0.685000,0.048109,-172.682911,-311.761842
38,0.654084,0.641999,0.953663,135093.000000,-1.765545,-1.953378,0.720152,-0.015538,-0.124109,0.830000,0.108571,-172.838918,-312.522469
57,0.625159,0.617800,0.953449,207042.000000,-1.764271,-1.951866,0.720374,-0.014436,-0.175237,0.935000,0.160801,-172.827894,-313.033746
76,0.557064,0.590786,0.952833,272372.000000,-1.762554,-1.950151,0.719952,-0.015967,-0.239071,0.960000,0.223104,-172.843206,-313.672088
95,0.602060,0.568186,0.952062,338901.000000,-1.761783,-1.948856,0.720075,-0.024467,-0.301406,0.955000,0.276939,-172.928206,-314.295433
114,0.576698,0.551877,0.951735,408759.000000,-1.760809,-1.947991,0.720271,-0.029311,-0.346567,0.970000,0.317256,-172.976647,-314.747048
133,0.528876,0.538361,0.951465,471605.000000,-1.760585,-1.947447,0.720337,-0.023469,-0.374775,0.990000,0.351306,-172.918221,-315.029122
152,0.531174,0.521069,0.951003,536736.000000,-1.759748,-1.946649,0.720443,-0.019517,-0.414651,0.985000,0.395133,-172.878709,-315.427880
171,0.542920,0.517502,0.950799,606261.000000,-1.759034,-1.945768,0.720056,-0.033803,-0.438573,1.000000,0.404770,-173.021566,-315.667102
190,0.489840,0.508934,0.950268,676170.000000,-1.758484,-1.945098,0.720374,-0.029069,-0.458000,0.990000,0.428931,-172.974222,-315.861373


Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.10s/it]
[W720 21:57:27.208571361 CUDACachingAllocator.cpp:3933] memory allocation failed with OOM on device 0 while trying to allocate 463470592 bytes (free: 96534528, total: 25428426752).
Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.21s/it]
[transformers] There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


🎉 DPO training completed!


In [ ]:
test_dataset_dpo

Dataset({
    features: ['prompt', 'chosen', 'rejected'],
    num_rows: 300
})

In [35]:
from transformers import AutoModelForCausalLM
from trl import DPOTrainer

sft_checkpoint = "./train/LiquidAI_LFM2_700M/checkpoint-1014"
dpo_checkpoint = "./train/dpo/LiquidAI_LFM2_700M/checkpoint-361"

policy_model = AutoModelForCausalLM.from_pretrained(
    dpo_checkpoint,
    device_map="auto",
    dtype="bfloat16",
    attn_implementation="sdpa",
)

ref_model = AutoModelForCausalLM.from_pretrained(
    sft_checkpoint,
    device_map="auto",
    dtype="bfloat16",
    attn_implementation="sdpa",
)

policy_model.config.use_cache = False
ref_model.config.use_cache = False

dpo_config.remove_unused_columns = False

test_trainer = DPOTrainer(
    model=policy_model,
    ref_model=ref_model,
    args=dpo_config,
    train_dataset=dataset_dpo["train"],
    eval_dataset=test_dataset_dpo,
    processing_class=tokenizer,
)

test_metrics = test_trainer.evaluate(metric_key_prefix="test")

print(test_metrics)

Tokenizing eval dataset: 100%|██████████| 300/300 [00:01<00:00, 234.53 examples/s]


Training Loss,Validation Loss,Step,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
No log,0.491711,0,0.942253,0.000000,-1.774505,-1.962077,0.721297,-0.033354,-0.508601,0.996667,0.475246,-174.310000,-316.190000


{'test_loss': 0.49171051383018494, 'eval_entropy': 0.9422526041666667, 'eval_num_tokens': 0.0, 'eval_logits/chosen': -1.774505168776013, 'eval_logits/rejected': -1.96207745684707, 'eval_mean_token_accuracy': 0.7212965127825737, 'eval_rewards/chosen': -0.03335427406442856, 'eval_rewards/rejected': -0.5086005559687813, 'eval_rewards/accuracies': 0.9966666666666667, 'eval_rewards/margins': 0.4752462811643879, 'eval_logps/chosen': -174.31, 'eval_logps/rejected': -316.19}


In [36]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

best_dpo_checkpoint = "./train/dpo/LiquidAI_LFM2_700M/checkpoint-361"

tokenizer = AutoTokenizer.from_pretrained(best_dpo_checkpoint)

model = AutoModelForCausalLM.from_pretrained(
    best_dpo_checkpoint,
    device_map="auto",
    dtype=torch.bfloat16,
    attn_implementation="sdpa",
)

model.eval()
model.config.use_cache = True

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 443.33it/s]


In [37]:
idx = 2
example = train_dataset_dpo[idx]

messages = [
    {
        "role": "user",
        "content": example["prompt"],
    }
]

input_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

inputs = tokenizer(
    input_text,
    return_tensors="pt",
).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=1024,
        do_sample=True,
        temperature=0.6,
        top_p=0.9,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.eos_token_id,
    )

generated_ids = outputs[0][inputs["input_ids"].shape[-1]:]
response = tokenizer.decode(generated_ids, skip_special_tokens=True)

print("PROMPT:")
print(example["prompt"])

print("\nGENERACIÓN:")
print(response)

print("\nCHOSEN:")
print(example["chosen"])

print("\nREJECTED:")
print(example["rejected"])

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


PROMPT:
Al amanecer,
Los cantos de los pájaros llenan el aire de melodías.
El aroma de las flores recién florecidas,
Se eleva en la suave brisa.

Los primeros rayos de luz,
Asoman entre los árboles.
El calor del sol,
Se extiende sobre la hierba con facilidad.

Al atardecer,
Los susurros del viento,
Callan el día en silencio.
El cielo se transforma en colores magníficos,
Enviando escalofríos por la espalda en la quietud.

El olor del rocío vespertino,
Se mezcla con el aire fresco.
El canto de los grillos y las ranas,
Evoca un sentido de cuidado calmante.

Los colores suaves del cielo,
Presentan un espectáculo sereno.
Una vista tan impresionante,
Los sentimientos de serenidad comienzan a fluir.

Al final del día,
La belleza de la naturaleza,
Trae alegría y paz a nuestro corazón.
Un momento precioso, para atesorar,
Y recordar cuando el mundo se desmorona. 


GENERACIÓN:
Vale la pena detenerse un segundo y escuchar cómo el aire fresco llena tus pulmones mientras observas ese atardecer que 

In [38]:
import os
from dotenv import load_dotenv
from huggingface_hub import login, HfApi
from transformers import AutoTokenizer, AutoModelForCausalLM

'''load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")

login(token=HF_TOKEN)

# Repo destino
hub_model_id = "thinkPy/dpo_LFM2_5_350M"'''

# Usa tu mejor checkpoint DPO
best_checkpoint_path = "./train/dpo/LiquidAI_LFM2_700M/checkpoint-361"
# Si tu checkpoint real tiene otro número, cámbialo aquí.

# Cargar modelo y tokenizer desde el checkpoint
model = AutoModelForCausalLM.from_pretrained(
    best_checkpoint_path,
    device_map="auto",
    dtype="bfloat16",
)

tokenizer = AutoTokenizer.from_pretrained(best_checkpoint_path)

# Recomendado para inferencia
model.config.use_cache = True

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 437.80it/s]


In [39]:
import os

'''model.push_to_hub(
    hub_model_id,
    token=HF_TOKEN,
    private=False,
)

tokenizer.push_to_hub(
    hub_model_id,
    token=HF_TOKEN,
    private=False,
)

print(f"✅ Modelo subido a: https://huggingface.co/{hub_model_id}")'''

'model.push_to_hub(\n    hub_model_id,\n    token=HF_TOKEN,\n    private=False,\n)\n\ntokenizer.push_to_hub(\n    hub_model_id,\n    token=HF_TOKEN,\n    private=False,\n)\n\nprint(f"✅ Modelo subido a: https://huggingface.co/{hub_model_id}")'